# Lesson 15: The Content Writer

The **Content Writer** is the first of 3 agents in our product — it researches topics and writes complete SEO articles.

In Lesson 9, you learned that tools let agents **act**. The Content Writer combines:
- **DataForSEO search** — to research topics (from Lesson 9)
- **Storage tools** — to save finished articles to disk
- **Clear instructions** — to write well-structured SEO content

One agent does the full job: research, write, save.

```
Topic --> [Content Writer] --> web_search + write + save_article --> Saved article
```

> **Note**: Claude (Anthropic) supports using `tools` and `output_schema` together. The Content Writer uses tools but outputs plain Markdown, so it doesn't need `output_schema` here.

## Why One Agent, Not Three?

You might think you need separate agents for research and writing. But Claude Sonnet is capable enough to do both in one step:

1. Search the web for the topic (1-2 searches)
2. Process the search results
3. Write a complete Markdown article
4. Save it to disk with `save_article()`

This is **simpler** (fewer files, fewer handoffs) and **faster** (one agent call instead of chaining 2-3 agents).

> **Cost:** ~$0.10-0.30 per article (Sonnet call with search + writing). Takes 30-90 seconds.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## The Content Writer — Production Code

Here's the actual agent definition from `output/backend/agents/content_writer.py`.

Notice the pattern:
- **Model**: Claude Sonnet (supports tools)
- **Tools**: DataForSEO search (conditional) + storage functions
- **Instructions**: Detailed writing guidelines (structure, length, SEO)

The search tool is **conditional** — if `DATA_FOR_SEO_API_KEY` isn't set, the writer still works but relies on its built-in knowledge.

In [ ]:
# Show the actual production code
content_writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(content_writer_path, "r", encoding="utf-8") as f:
    print(f.read())

## Building the Content Writer Step by Step

Let's build the same agent from scratch to understand each piece.

### Step 1: Set up search tools (conditional)

In [ ]:
from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials
from tools.storage import save_article, list_all_articles

# Conditional search: works with or without DataForSEO
tools = [save_article, list_all_articles]
creds = get_dataforseo_credentials()
if creds:
    tools.insert(0, DataForSEOSearchTools(login=creds[0], password=creds[1]))
    print("Search tools: DataForSEO (web search enabled)")
else:
    print("Search tools: None (will write from built-in knowledge)")

print(f"Total tools: {len(tools)}")

### Step 2: Create the agent with writing instructions

In [ ]:
writer = Agent(
    name="Content Writer",
    role="Research topics and write SEO articles",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=tools,
    instructions=[
        "You research topics and write comprehensive SEO articles.",
        "RESEARCH: Do 1-2 web searches max per article.",
        "Write a well-structured Markdown article with:",
        "  - An H1 title",
        "  - 5-8 H2 section headings",
        "  - H3 subheadings where appropriate",
        "  - Bolded important keywords naturally within the text",
        "  - Bullet/numbered lists where helpful",
        "  - 1500-2500 words of engaging content",
        "After writing, call save_article with the topic, full article text, and target keywords.",
        "Never use emojis or icons.",
    ],
    markdown=True,
)

print(f"Agent: {writer.name}")
print(f"Model: Claude Sonnet")
print(f"Tools: {len(tools)}")

### Step 3: Run the writer

When you give the writer a topic, it:
1. Searches the web (if DataForSEO is available)
2. Writes a complete Markdown article
3. Calls `save_article()` to save it to `content/` directory

The saved article goes to `content/{slug}.md` and metadata to `content/articles.json`.

In [ ]:
topic = "How to optimize on-page SEO for your website"

print(f"Writing article about: {topic}")
print("This takes 30-90 seconds...\n")

response = writer.run(f"Write an SEO article about: {topic}")
print(response.content[:2000])
if len(response.content) > 2000:
    print("\n... (truncated for display)")

In [ ]:
# Check: was the article saved?
import json
result = list_all_articles()
articles = json.loads(result)
print(f"Articles in storage: {len(articles)}")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['word_count']} words, {a['status']})")

## The ContentOutline Schema (Bonus)

The Content Writer outputs plain Markdown. But structured output is still useful for other purposes — like when you want just an outline.

Here's the same `ContentOutline` schema from Lesson 11's structured output, showing how you'd use it if you wanted **just** a structured outline before writing:

```python
class OutlineSection(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    key_points: list[str] = Field(description="Bullet points to cover")

class ContentOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    meta_description: str = Field(description="Meta description, max 160 chars")
    target_keywords: list[str] = Field(description="Primary SEO keywords")
    sections: list[OutlineSection] = Field(description="Content sections")
```

The Content Writer doesn't use this schema (it writes the full article directly). But it's a useful pattern when you need structure before content.

## Summary

What you learned:
- The **Content Writer** researches and writes articles in one agent
- Uses **DataForSEO search** (conditional) + **storage tools** (save, list)
- Produces complete **Markdown articles** with SEO structure
- Articles are **saved locally** to `content/` directory
- **Simpler than chaining** — one capable agent beats multiple simple ones

**Next lesson:** Image Finder and AIO Analyzer — the 2 supporting agents.

## Exercise

Change the `topic` variable to a topic relevant to your work, then re-run the writer cell above.

After the article is saved, write code below to:
1. List all articles and count them
2. Get the content of the most recent article (use `get_article_content` from `tools.storage`)
3. Print the first 500 characters of the article content

In [ ]:
# Exercise: Fill in the blanks to inspect your article
from tools.storage import get_article_content
import json

# 1. List all articles
result = ___()
articles = json.loads(result)
print(f"Total articles: {___(articles)}")

# 2. Get the most recent article's content (last one in the list)
last_id = articles[-1][___]
content_json = get_article_content(last_id)
content = json.loads(content_json)

# 3. Print first 500 characters
print(f"\nArticle: {content['topic']}")
print(content['article_markdown'][:___])

<details>
<summary>Click to reveal answer</summary>

```python
from tools.storage import get_article_content
import json

# 1. List all articles
result = list_all_articles()
articles = json.loads(result)
print(f"Total articles: {len(articles)}")

# 2. Get the most recent article's content
last_id = articles[-1]["id"]
content_json = get_article_content(last_id)
content = json.loads(content_json)

# 3. Print first 500 characters
print(f"\nArticle: {content['topic']}")
print(content['article_markdown'][:500])
```
</details>